# HLS, Red Boost, HSV Legacy и HSV Narrow

Четыре цветовых метода сравниваются на одних и тех же кадрах. Каждый метод выбирает две крупнейшие компоненты маски. Сетап используется только для группировки метрик. Начальные пороги задаются в `config.local.yaml`.

In [ ]:
import ipywidgets as widgets
import plotly.io as pio
from IPython.display import display

from laser_detection.benchmarking import ImageDataError, benchmark_dataset, read_image, timing_summary
from laser_detection.config import load_config
from laser_detection.data import load_dataset
from laser_detection.detection_plots import detection_figures, diagnostic_png
from laser_detection.detectors import baseline_detectors
from laser_detection.evaluation import (
    comparison_table, dataset_split, detection_metrics, load_detection_comparison,
    localization_disagreements, score_predictions, worst_cases,
)
from laser_detection.export import export_detection

pio.renderers.default = "plotly_mimetype"
config = load_config()

In [ ]:
frame = load_dataset(config)
if not config.images_dir.is_dir():
    raise FileNotFoundError(f"Укажите доступный каталог фотографий: {config.images_dir}")

splits = dataset_split(frame, config)
display(splits.groupby("split", sort=False).size().rename("Кадры"))
detectors = baseline_detectors(config.detection)
annotations = frame.set_index("image")

## Предварительный просмотр

При разбиении по сериям предварительный просмотр использует обучающие кадры. Выберите один из первых трёх кадров. Изображение читается по запросу.

In [ ]:
def frame_viewer(options):
    picker = widgets.Combobox(
        options=tuple(options), value=options[0], ensure_option=True, description="Кадр",
    )
    show = widgets.Button(description="Показать")
    picture = widgets.Image(format="png")
    message = widgets.HTML()

    def update(_=None):
        picture.value = b""
        message.value = ""
        name = picker.value
        if name not in annotations.index:
            message.value = "Выберите имя известного кадра."
            return
        try:
            image = read_image(name, config)
        except ImageDataError:
            message.value = "Кадр не читается или его размер не совпадает с настройками."
            return
        picture.value = diagnostic_png(image, annotations.loc[name], detectors)

    show.on_click(update)
    display(widgets.VBox([widgets.HBox([picker, show]), message, picture]))
    update()


preview_split = "train" if config.series_csv else "exploratory"
names = splits.loc[splits["split"].eq(preview_split), "image"].head(3).tolist()
frame_viewer(names)

## Последовательный замер

Изображения читаются по одному. Методы выполняются последовательно, после прогрева. Первая позиция циклически переходит к следующему методу на каждом доступном кадре. Чтение изображения и визуализация исключены из времени алгоритма.

In [ ]:
predictions, environment = benchmark_dataset(frame, detectors, config)
scored = score_predictions(predictions, frame, splits)
display(scored.groupby(["method", "status"]).size().rename("Кадры"))
if not scored["status"].eq("ok").any():
    raise ValueError("Нет пригодных изображений. Проверьте пути, форматы и исходный размер.")

## Точность и скорость

Сначала сравните результаты на train/validation. Используйте test для окончательной оценки после фиксации порогов. Для исследовательского запуска без серий доступен только exploratory. Recall@t учитывает промахи и требует, чтобы обе точки уложились в t пикселей исходного кадра. Статистики координат используют только кадры с разметкой и предсказанной парой.

In [ ]:
FINAL_TEST = False  # Включите один раз после фиксации порогов.
if config.series_csv:
    report_split = "test" if FINAL_TEST else "validation"
else:
    report_split = "exploratory"

reported = scored if FINAL_TEST else scored.loc[scored["split"].ne("test")]
metrics = detection_metrics(reported)
by_setup = detection_metrics(reported, by_setup=True)
timings = timing_summary(scored)
figures = detection_figures(metrics, timings, report_split)

display(comparison_table(metrics, timings, report_split).rename(columns={
    "method": "Метод", "recall": "Recall", "recall_at_5px": "Recall@5px",
    "recall_at_10px": "Recall@10px", "recall_at_20px": "Recall@20px",
    "median_point_error_px": "Median error (px)", "total_mean_ms": "Mean time (ms)",
}))
display(metrics.loc[metrics["split"].eq(report_split)])
display(by_setup.loc[by_setup["split"].eq(report_split)])
display(timings)
print(environment["cpu"], "| OpenCV", environment["opencv_version"],
      "| threads:", environment["opencv_threads"], "| warmup:", environment["warmup_frames_completed"])
for figure in figures.values():
    figure.show()

## Ошибки по кадрам

Таблицы показывают пропущенные пары, ложные срабатывания и совпавшие пары с наибольшей ошибкой. Введите или выберите имя кадра, чтобы сравнить маски и точки обоих методов.

In [ ]:
error_columns = ["image", "setup_id", "outcome", "error1_px", "error2_px", "pair_max_error_px"]
for detector in detectors:
    print(detector.name)
    display(worst_cases(scored, detector.name, report_split)[error_columns])

diagnostic_names = scored.loc[
    scored["split"].eq(report_split) & scored["status"].eq("ok"), "image"
].drop_duplicates().tolist()
if diagnostic_names:
    frame_viewer(diagnostic_names)
else:
    print("Нет доступных кадров для выбранной части данных.")

## Где HSV Legacy и Red Boost расходятся

Кадр входит в список метода, если обе точки укладываются в допуск у этого метода, а у второго есть промах или хотя бы одна точка за пределами допуска. Сравниваются только доступные размеченные кадры выбранной части данных.

In [ ]:
tolerance = 10
disagreements = localization_disagreements(scored, "hsv_legacy", "red_boost", report_split, tolerance)
for method in ("hsv_legacy", "red_boost"):
    cases = disagreements.loc[disagreements["better_method"].eq(method)]
    print(f"{method}: {len(cases)} кадров при допуске {tolerance} px")
    if not cases.empty:
        display(cases.head(10))
        frame_viewer(cases["image"].tolist())

## Локальный экспорт

Установите `EXPORT = True`. Предсказания содержат все кадры и признаки разбиения, включая ошибку чтения, если она возникла. Не используйте тестовые строки для настройки порогов. Перед коммитом очистите выводы ноутбука. Каждый экспорт создаёт новую папку. Для следующего запуска измените `RUN_NAME`.

In [ ]:
EXPORT = False
RUN_NAME = "hsv_comparison"
if EXPORT:
    for path in export_detection(
        scored, splits, metrics, by_setup, timings, environment, figures, config, run_name=RUN_NAME,
    ):
        print(path)

## Сравнение из сохранённых CSV

Запустите первую ячейку с импортами и конфигурацией, затем ячейку ниже. Укажите имя сохранённого запуска и часть данных. Для старого двухметодного экспорта укажите `SAVED_RUN = "."`.

In [ ]:
SAVED_RUN = None  # Например, "hsv_comparison"
SAVED_SPLIT = "exploratory"  # Или "validation" / "test" при разбиении по сериям
if SAVED_RUN is not None:
    display(load_detection_comparison(config.detection_output_dir / SAVED_RUN, SAVED_SPLIT))